# 04 — Tool Calling Fundamentals with HubSpot CRM

**Chapter 2 | Mastering Agentic AI for Marketing Technology**

This notebook teaches OpenAI function calling (tool use) by building CRM tools
that an LLM can invoke autonomously. We implement four HubSpot operations:

- `get_contact` — Retrieve a contact by email
- `create_contact` — Create a new CRM contact
- `search_contacts` — Search contacts by property
- `update_deal_stage` — Move a deal through the pipeline

Each tool has a real HubSpot API implementation and a mock fallback.

In [ ]:
import os
import json
import logging
from datetime import datetime
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("tool_calling")

USE_MOCK = os.getenv("USE_MOCK_APIS", "true").lower() == "true"
HUBSPOT_API_KEY = os.getenv("HUBSPOT_API_KEY", "")

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY", "mock-key"),
    base_url=os.getenv("OPENAI_BASE_URL"),
)
MODEL = os.getenv("OPENAI_MODEL", "gpt-4.1")

print(f"Mock mode: {USE_MOCK}")
print(f"HubSpot key present: {bool(HUBSPOT_API_KEY)}")

## Mock CRM Database

When `USE_MOCK_APIS=true`, these tools use an in-memory database
with realistic marketing data.

In [ ]:
MOCK_CONTACTS = {
    "sarah@acmecorp.com": {
        "id": "101", "email": "sarah@acmecorp.com",
        "firstname": "Sarah", "lastname": "Johnson",
        "company": "Acme Corp", "jobtitle": "VP Marketing",
        "lifecyclestage": "opportunity", "lead_score": 85,
        "last_activity": "2025-01-15T10:30:00Z"
    },
    "mike@techstart.io": {
        "id": "102", "email": "mike@techstart.io",
        "firstname": "Mike", "lastname": "Chen",
        "company": "TechStart", "jobtitle": "Growth Lead",
        "lifecyclestage": "lead", "lead_score": 42,
        "last_activity": "2025-01-20T14:15:00Z"
    },
    "emma@globalretail.com": {
        "id": "103", "email": "emma@globalretail.com",
        "firstname": "Emma", "lastname": "Davis",
        "company": "Global Retail", "jobtitle": "CMO",
        "lifecyclestage": "customer", "lead_score": 95,
        "last_activity": "2025-01-22T09:00:00Z"
    }
}

MOCK_DEALS = {
    "deal_201": {"id": "deal_201", "name": "Acme Corp — Enterprise", "stage": "proposal", "amount": 48000, "contact": "101"},
    "deal_202": {"id": "deal_202", "name": "TechStart — Growth", "stage": "discovery", "amount": 12000, "contact": "102"},
}

print(f"Mock database: {len(MOCK_CONTACTS)} contacts, {len(MOCK_DEALS)} deals")

## Define CRM Tool Functions

Each function has a real API path and a mock fallback.

In [ ]:
def get_contact(email: str) -> str:
    """Retrieve a HubSpot contact by email address."""
    logger.info(f"[{datetime.now().isoformat()}] get_contact called: {email}")

    if not USE_MOCK and HUBSPOT_API_KEY:
        from hubspot import HubSpot
        hs = HubSpot(access_token=HUBSPOT_API_KEY)
        try:
            result = hs.crm.contacts.basic_api.get_by_id(
                contact_id=email, id_property="email",
                properties=["email", "firstname", "lastname", "company", "jobtitle", "lifecyclestage"]
            )
            return json.dumps(result.properties)
        except Exception as e:
            return json.dumps({"error": str(e)})

    # MOCK MODE
    contact = MOCK_CONTACTS.get(email)
    if contact:
        return json.dumps(contact)
    return json.dumps({"error": f"No contact found for {email}"})


def create_contact(email: str, firstname: str, lastname: str, company: str) -> str:
    """Create a new contact in HubSpot CRM."""
    logger.info(f"[{datetime.now().isoformat()}] create_contact called: {email}")

    if not USE_MOCK and HUBSPOT_API_KEY:
        from hubspot import HubSpot
        from hubspot.crm.contacts import SimplePublicObjectInputForCreate
        hs = HubSpot(access_token=HUBSPOT_API_KEY)
        try:
            result = hs.crm.contacts.basic_api.create(
                simple_public_object_input_for_create=SimplePublicObjectInputForCreate(
                    properties={"email": email, "firstname": firstname, "lastname": lastname, "company": company}
                )
            )
            return json.dumps({"id": result.id, "created": True})
        except Exception as e:
            return json.dumps({"error": str(e)})

    # MOCK MODE
    new_id = f"{200 + len(MOCK_CONTACTS)}"
    MOCK_CONTACTS[email] = {
        "id": new_id, "email": email, "firstname": firstname,
        "lastname": lastname, "company": company,
        "lifecyclestage": "subscriber", "lead_score": 0,
        "last_activity": datetime.now().isoformat()
    }
    return json.dumps({"id": new_id, "created": True})


def search_contacts(property_name: str, value: str) -> str:
    """Search HubSpot contacts by a property value."""
    logger.info(f"[{datetime.now().isoformat()}] search_contacts: {property_name}={value}")

    if not USE_MOCK and HUBSPOT_API_KEY:
        from hubspot import HubSpot
        hs = HubSpot(access_token=HUBSPOT_API_KEY)
        try:
            result = hs.crm.contacts.search_api.do_search(
                public_object_search_request={
                    "filterGroups": [{"filters": [{"propertyName": property_name, "operator": "CONTAINS_TOKEN", "value": value}]}],
                    "properties": ["email", "firstname", "lastname", "company"]
                }
            )
            return json.dumps([r.properties for r in result.results])
        except Exception as e:
            return json.dumps({"error": str(e)})

    # MOCK MODE
    matches = [c for c in MOCK_CONTACTS.values() if value.lower() in str(c.get(property_name, "")).lower()]
    return json.dumps(matches)


def update_deal_stage(deal_id: str, new_stage: str) -> str:
    """Update a deal's pipeline stage in HubSpot."""
    logger.info(f"[{datetime.now().isoformat()}] update_deal_stage: {deal_id} -> {new_stage}")

    if not USE_MOCK and HUBSPOT_API_KEY:
        from hubspot import HubSpot
        from hubspot.crm.deals import SimplePublicObjectInput
        hs = HubSpot(access_token=HUBSPOT_API_KEY)
        try:
            result = hs.crm.deals.basic_api.update(
                deal_id=deal_id,
                simple_public_object_input=SimplePublicObjectInput(properties={"dealstage": new_stage})
            )
            return json.dumps({"id": result.id, "stage": new_stage, "updated": True})
        except Exception as e:
            return json.dumps({"error": str(e)})

    # MOCK MODE
    if deal_id in MOCK_DEALS:
        old_stage = MOCK_DEALS[deal_id]["stage"]
        MOCK_DEALS[deal_id]["stage"] = new_stage
        return json.dumps({"id": deal_id, "old_stage": old_stage, "new_stage": new_stage, "updated": True})
    return json.dumps({"error": f"Deal {deal_id} not found"})


# Map function names to callables
TOOL_FUNCTIONS = {
    "get_contact": get_contact,
    "create_contact": create_contact,
    "search_contacts": search_contacts,
    "update_deal_stage": update_deal_stage,
}

print("CRM tools defined:", list(TOOL_FUNCTIONS.keys()))

## Define Tool Schemas for OpenAI

These JSON schemas tell the LLM what tools are available and how to call them.

In [ ]:
TOOL_SCHEMAS = [
    {
        "type": "function",
        "function": {
            "name": "get_contact",
            "description": "Retrieve a contact from HubSpot CRM by email address. Returns contact properties.",
            "parameters": {
                "type": "object",
                "properties": {"email": {"type": "string", "description": "The contact's email address"}},
                "required": ["email"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "create_contact",
            "description": "Create a new contact in HubSpot CRM.",
            "parameters": {
                "type": "object",
                "properties": {
                    "email": {"type": "string", "description": "Email address"},
                    "firstname": {"type": "string", "description": "First name"},
                    "lastname": {"type": "string", "description": "Last name"},
                    "company": {"type": "string", "description": "Company name"}
                },
                "required": ["email", "firstname", "lastname", "company"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_contacts",
            "description": "Search HubSpot contacts by a property (e.g., company, lifecyclestage, jobtitle).",
            "parameters": {
                "type": "object",
                "properties": {
                    "property_name": {"type": "string", "description": "CRM property to search (e.g., 'company', 'lifecyclestage')"},
                    "value": {"type": "string", "description": "Value to search for"}
                },
                "required": ["property_name", "value"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "update_deal_stage",
            "description": "Update a deal's pipeline stage in HubSpot (e.g., discovery, proposal, closed-won).",
            "parameters": {
                "type": "object",
                "properties": {
                    "deal_id": {"type": "string", "description": "The HubSpot deal ID"},
                    "new_stage": {"type": "string", "description": "New pipeline stage"}
                },
                "required": ["deal_id", "new_stage"]
            }
        }
    }
]

print(f"Registered {len(TOOL_SCHEMAS)} tool schemas")

## The Agent Loop

This is the core pattern: send a message → LLM decides to call a tool →
we execute the tool → feed the result back → LLM generates the final answer.

In [ ]:
def run_crm_agent(user_query: str) -> str:
    """Run a CRM agent that can use HubSpot tools via function calling."""
    messages = [
        {"role": "system", "content": (
            "You are a CRM assistant with access to HubSpot tools. "
            "Use the available tools to look up contacts, create records, "
            "search the CRM, and update deals. Always explain what you found."
        )},
        {"role": "user", "content": user_query}
    ]

    if USE_MOCK:
        # Simulate tool calling flow
        if "sarah" in user_query.lower() or "acme" in user_query.lower():
            result = get_contact("sarah@acmecorp.com")
            return f"I looked up the contact and found:\n{result}"
        elif "search" in user_query.lower() or "find" in user_query.lower():
            result = search_contacts("company", "Tech")
            return f"Search results:\n{result}"
        elif "create" in user_query.lower():
            result = create_contact("new@example.com", "New", "Contact", "Example Inc")
            return f"Contact created:\n{result}"
        elif "deal" in user_query.lower():
            result = update_deal_stage("deal_201", "closed-won")
            return f"Deal updated:\n{result}"
        return "I can help with CRM lookups, contact creation, searches, and deal updates."

    # Real API flow with tool calling loop
    max_iterations = 5
    for _ in range(max_iterations):
        response = client.chat.completions.create(
            model=MODEL, messages=messages, tools=TOOL_SCHEMAS
        )
        msg = response.choices[0].message

        if not msg.tool_calls:
            return msg.content

        messages.append(msg)
        for tc in msg.tool_calls:
            fn = TOOL_FUNCTIONS[tc.function.name]
            args = json.loads(tc.function.arguments)
            logger.info(f"Calling {tc.function.name}({args})")
            result = fn(**args)
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})

    return "Max iterations reached."

print("Agent loop defined.")

## Demo: Run the Agent

Try different queries to see the agent use different CRM tools.

In [ ]:
queries = [
    "Look up Sarah's contact info at Acme Corp",
    "Search for all contacts at tech companies",
    "Create a new contact: John Smith at DataCo, john@dataco.com",
    "Update the Acme deal to closed-won",
]

for q in queries:
    print(f"\n{'='*60}")
    print(f"Query: {q}")
    print(f"{'='*60}")
    result = run_crm_agent(q)
    print(result)

## Key Takeaways

1. **Tool schemas** tell the LLM what functions are available and their parameters
2. **The agent loop** handles the back-and-forth: LLM → tool call → result → LLM
3. **Mock fallbacks** let you develop and test without API keys
4. **Logging** every tool call is essential for observability in production

This tool calling pattern is the foundation for every project in this book.
In Chapter 3, we'll wrap these tools in the OpenAI Agents SDK for a more
structured approach with guardrails and handoffs.

**Next:** Project 1 — Customer Journey Digital Twin (Chapter 3)